<a href="https://colab.research.google.com/github/akashkguw/orion/blob/main/orion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orion: Sparse Attention for Long-Context Transformers

This notebook demonstrates training decoder-only Transformers with sparse attention (window + expander edges) for efficient long-context modeling.

**Key Features:**
- Sparse Attention - O(T*(W+d)) complexity vs O(T^2) for dense (7x faster on 512 tokens)
- Reproducible - Deterministic training with seed control
- Configurable - YAML-based configs for easy experimentation
- Well-tested - 82+ tests covering all components

## Setup: Install Dependencies

In [ ]:
!git clone https://github.com/akashkguw/orion.git
%cd /content/orion
!ls
!pip -q install -r requirements.txt -r requirements-dev.txt

In [ ]:
import importlib.util

print("torch installed:", importlib.util.find_spec("torch") is not None)

In [ ]:
# If Torch is not available, uncomment and run:
# !pip -q install torch

## What is Sparse Attention?

Sparse attention reduces computational complexity from O(T^2) to O(T*(W+d)) by attending to only a subset of positions:

- **Local Window** - Dense context for nearby tokens (short-range dependencies)
- **Expander Edges** - Structured long-range connections using modular arithmetic

### Example Pattern

For query at position q=10 with window_size=4, expander_degree=3:

```
Window:   [7, 8, 9, 10]           (local context)
Expander: [9, 6, 1]               (long-range via quadratic residues)
Union:    [1, 6, 7, 8, 9, 10]     (7 positions vs 11 for dense)
```

### Performance

| Metric | Dense | Sparse | Speedup |
|--------|-------|--------|----------|
| Complexity | O(T^2) | O(T*(W+d)) | ~7x |
| Memory | O(T^2*Dh) | O(T*(W+d)*Dh) | ~7x |
| Example (T=512, W=64, d=8) | 262K pos/query | 36.8K pos/query | 7.1x |

## Quick Start: Train with Dense Attention

### Execute Training (Smoke Test - 20 steps)

In [ ]:
!python -m orion.train --config configs/golden.yaml

### Evaluate Model

In [ ]:
!python -m orion.eval --config configs/golden.yaml --checkpoint runs/latest/checkpoint.pt

### View Training Logs

In [ ]:
!tail -n 5 runs/latest/metrics.jsonl

## Train with Sparse Attention

### Sparse Attention Configuration

Sparse attention is configured via YAML:

```yaml
model:
  attention_type: sparse
  window_size: 64        # Local window size
  expander_degree: 8     # Long-range neighbors
```

**When to use sparse attention:**
- Long sequences (512+)
- Limited GPU memory
- Need stable training with diverse patterns
- Short sequences (<256) - dense is simpler

### Sparse Attention Smoke Test (5 steps)

In [ ]:
!python -m orion.train --config configs/exp_orion_sparse_smoke.yaml

### Evaluate Sparse Attention Model

In [ ]:
!python -m orion.eval --config configs/exp_orion_sparse_smoke.yaml --checkpoint runs/exp_orion_sparse_smoke/checkpoint.pt

## Train with Dense Attention (Full Dataset)

### Dense Attention on Shakespeare Dataset

Uses standard dense attention (O(T^2)) for comparison. Good for understanding baseline performance.

In [ ]:
!python -m orion.train --config configs/tinyshakespeare_dense.yaml

## Train with Sparse Attention (Full Dataset)

### Sparse Attention on Shakespeare Dataset

Uses sparse attention (O(T*(W+d))) for efficient long-context modeling. Compare performance with dense attention above.

In [ ]:
!python -m orion.train --config configs/tinyshakespeare_sparse.yaml

## Comparison: Dense vs Sparse Attention

### Available Configurations

| Config | Attention | Window | Expander | Use Case |
|--------|-----------|--------|----------|----------|
| golden.yaml | Dense | - | - | Quick smoke test |
| tinyshakespeare_dense.yaml | Dense | - | - | Baseline comparison |
| exp_orion_sparse_smoke.yaml | Sparse | 64 | 8 | Quick sparse test |
| tinyshakespeare_sparse.yaml | Sparse | 64 | 8 | Full sparse training |
| exp_sparse_window_256.yaml | Sparse | 256 | 8 | Larger window |

### Performance Metrics

After training, compare:
- Loss - Lower is better
- Perplexity - Lower is better
- Training Time - Sparse should be faster
- Memory Usage - Sparse should use less VRAM

## Advanced: Custom Configuration

### Create Custom Sparse Attention Config

You can create custom configs by modifying YAML files. Example:

```yaml
run:
  out_dir: runs/my_experiment
  seed: 42
  steps: 500
  log_every: 10
  save_every: 100

data:
  dataset: tinyshakespeare
  seq_len: 512          # Longer sequences benefit from sparse
  batch_size: 8

model:
  name: tiny
  d_model: 256
  n_layers: 4
  n_heads: 4
  attention_type: sparse
  window_size: 128      # Larger window for more local context
  expander_degree: 16   # More long-range neighbors

optim:
  lr: 3e-4
  weight_decay: 0.1
```

## Resources and Documentation

- README: Full documentation with examples
- SPARSE_ATTENTION_ARCHITECTURE.md: Deep technical dive into sparse attention
- GitHub: https://github.com/akashkguw/orion

### Key Files

- orion/attention/sparse.py - Sparse attention implementation
- orion/attention/dense.py - Dense attention implementation
- configs/ - Training configurations
- tests/test_sparse_attention.py - 82+ tests for sparse attention

## Troubleshooting

### Out of Memory (OOM)
- Use sparse attention instead of dense
- Reduce batch_size in config
- Reduce seq_len in config
- Reduce d_model in config

### Slow Training
- Check GPU is being used: !nvidia-smi
- Use sparse attention for long sequences
- Increase batch_size if memory allows

### Training Not Converging
- Try different seed values
- Adjust learning rate (lr in config)
- Increase training steps (steps in config)
- Check data loading with !head -c 100 data/tinyshakespeare.txt